# Compare: Generation Strategy Smoke Tests

This notebook runs quick, low-cost checks of the different generation strategies
implemented in `Compare/mitigation` using a small HuggingFace model.

You can change `MODEL_ID` (and other ids) below to try larger models.

In [17]:
MODEL_ID = "google/gemma-3-270m-it"  # small, CPU-friendly
AMATEUR_ID = "google/gemma-3-1b-it"  # reuse as amateur for CD demo
LANGUAGE = "Python"

TEST_INSTRUCTION = "What Python packages are useful for training a small neural network?"

In [18]:
import sys
sys.path.append("..")

from mitigation.interface import ChatMessage, ChatRole

def run_chat(gen, instruction: str, language: str = LANGUAGE) -> str:
    messages = [
        ChatMessage(role=ChatRole.SYSTEM, content=f"You are a coding assistant for {language} packages."),
        ChatMessage(role=ChatRole.USER, content=instruction),
    ]
    return gen.chat_generation(messages)

## Baseline

The baseline is the basic using of the model without any mitigation strategy.

In [19]:
import sys
sys.path.append("..")

from mitigation.baseline import BaselineGenerator

baseline_cfg = {"max_new_tokens": 32, 
                "temperature": 0.7,
                "generation_kwargs": {
                    "do_sample": True,
                    "top_p": 0.9,
                    "top_k": 0,
                    "num_beams": 1,
                }
                }

baseline = BaselineGenerator(model_name=MODEL_ID, config=baseline_cfg)
print(run_chat(baseline, TEST_INSTRUCTION))


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1713.19it/s, Materializing param=model.norm.weight]                                
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


system: You are a coding assistant for Python packages.
user: What Python packages are useful for training a small neural network?
assistant: 
The Python packages that are useful for training a small neural network are:
- TensorFlow
- PyTorch
- scikit-learn
- NumPy



## Overview of generation strategies

This notebook compares several decoding / mitigation strategies implemented under `Compare/mitigation`:

- **Baseline (`BaselineGenerator`)**: plain HuggingFace generation with configurable sampling (`temperature`, `top_p`, `top_k`, etc.).
- **Greedy (`GreedyGenerator`)**: deterministic argmax decoding, no sampling; good as a simple baseline.
- **DoLa (`DoLaGenerator`)**: Decoding by Contrasting Layers; reweights logits using differences between early and mature transformer layers.
- **Nudging (`NudgingGenerator`)**: uses a base and a "nudging" model; when the base is uncertain, the nudging model steers the next token.
- **RAG (`RagGenerator`)**: retrieves package-related statements from a vector DB and augments the prompt before generation.
- **Self-refine (`SelfRefineGenerator`)**: iteratively generates, validates package existence, and asks the model to correct invalid packages.
- **Contrastive decoding (`ContrastiveDecodingGenerator`)**: runs an expert and an amateur model in parallel and prefers tokens where the expert is confident and the amateur is not.

## Greedy decoding

`GreedyGenerator` uses the same model as the baseline but **always picks the highest-probability token** at each step (`do_sample=False`).

- **Key config fields**:
  - `max_new_tokens`: maximum length of the continuation.
  - `generation_kwargs.top_k`, `generation_kwargs.top_p`: kept mostly for completeness; with `do_sample=False` they have no effect.

Use this to understand how much randomness in the baseline sampling changes the package recommendations.

In [20]:
from mitigation.greedy import GreedyGenerator

greedy_cfg = {"max_new_tokens": 32, 
                "temperature": 0.0, 
                "generation_kwargs": {
                    "do_sample": False
                    }
            }

greedy = GreedyGenerator(model_name=MODEL_ID, config=greedy_cfg)
print("Greedy:\n")
print(run_chat(greedy, TEST_INSTRUCTION))

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 2263.40it/s, Materializing param=model.norm.weight]                                


Greedy:

system: You are a coding assistant for Python packages.
user: What Python packages are useful for training a small neural network?
assistant: 
Python packages that are useful for training a small neural network include:
- TensorFlow
- PyTorch
- scikit-learn
- NumPy
-


## DoLa decoding

`DoLaGenerator` implements *Decoding by Contrasting Layers*.
The idea is to compare logits from early ("premature") layers and a mature layer;
this can reduce hallucinations by preferring tokens whose probability grows
monotonically across layers.

- **Key config fields**:
  - `max_new_tokens`: length of continuation.
  - `dola_mature_layer`: index of the mature layer (often the last transformer block).
  - `dola_early_exit_layers`: comma-separated early layer indices to contrast against.
  - `dola_relative_top`, `dola_repetition_penalty`: control filtering and repetition penalty.

In [21]:
from mitigation.dola import DoLaGenerator
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
hidden_layers = model.config.num_hidden_layers
# hidden_layers = model.config.n_layers

dola_cfg = {
    "max_new_tokens": 32,
    "dola_mature_layer": hidden_layers,
    "dola_early_exit_layers": "4,8,12,16", # the gemma model has 18 layers
#    "dola_relative_top": 0.1,
#    "dola_repetition_penalty": 1.2
}

dola = DoLaGenerator(model_name=MODEL_ID, config=dola_cfg)
print("DoLa:\n")
print(run_chat(dola, TEST_INSTRUCTION))

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 2373.70it/s, Materializing param=model.norm.weight]                                


DoLa:

*   `TensorFlow`
*   `PyTorch`
*   `Scikit-learn`
*   `Keras`

What


## Nudging

`NudgingGenerator` uses **two models**:
- a *base* model that generates tokens normally, and
- a *nudging* model that intervenes when the base model is uncertain.

On each step, if the base model's top-1 probability is below `top_prob_thres`,
the nudging model's token is used instead. This can reduce hallucinations by
letting a more specialised model veto low-confidence steps.

- **Key config fields**:
  - `max_new_tokens`: length of continuation.
  - `top_prob_thres`: confidence threshold; lower → fewer nudging interventions.
  - `temperature`, `nudging_temperature`: sampling temperature for base / nudging.

In [22]:
from mitigation.nudging import NudgingGenerator

nudging_cfg = {
    "max_new_tokens": 32,
    "top_prob_thres": 0.4,
}

# For a quick demo, we use the same model as both base and nudging.
nudging = NudgingGenerator(
    base_model_name=MODEL_ID,
    nudging_model_name=MODEL_ID,
    config=nudging_cfg,
)

print("Nudging (base == nudging model, demo only):\n")
print(run_chat(nudging, TEST_INSTRUCTION))

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 2174.09it/s, Materializing param=model.norm.weight]                                


Nudging (base == nudging model, demo only):

Python packages that are useful for training a small neural network include:
- TensorFlow
- PyTorch
- scikit-learn
- NumPy
-


## RAG (Retrieval-Augmented Generation)

`RagGenerator` wraps another generator (e.g. `BaselineGenerator`) and augments
its input with retrieved statements from a vector database built over
`Compare/data/rag/RAG_data.jsonl`.

- **Pipeline**:
  1. Use `RagRetriever` (via `RagGenerator`) to retrieve the top-`k` relevant
     statements for the user question.
  2. Build a system prompt that constrains the output to package lists.
  3. Build a user prompt that includes both the original question and the
     retrieved statements.
  4. Call the inner generator's `chat_generation` with these messages.

- **Key config fields**:
  - `k`: number of statements to retrieve per question.

In [23]:
from mitigation.rag import RagGenerator

rag_cfg = {"k": 3}

rag = RagGenerator(inner=baseline, language=LANGUAGE, config=rag_cfg)
print("RAG + Baseline:\n")
print(run_chat(rag, TEST_INSTRUCTION))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2028.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG + Baseline:



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


system: You are a coding assistant that recommends Python packages to help answer questions. Use the provided statements to help form your response, but do not limit your response to those statements. Respond with only a list of Python packages, separated by commas and no additional text or formatting. Your response must begin with the name of a Python package.
user: What Python packages would be useful in solving the following coding question: What Python packages are useful for training a small neural network?

Here are some statements that may help answer the question:
The creme package could answer the question: "How can I use Python to build a simple neural network for classification tasks?" The torch package could answer the question: "How can I implement a neural network in Python with efficient GPU acceleration, allowing me to train large models quickly and accurately?" The torch package could answer the question: "How can I implement a neural network in Python with efficient G

## Self-refine

`SelfRefineGenerator` wraps an inner chat-capable generator and adds an
*iterative refinement loop* for package recommendations:

1. Generate an initial list of packages for the instruction.
2. Ask the model (in a separate validation turn) to mark each package as
   `Yes` (real package) or `No` (non-existent / invalid).
3. If any packages are invalid, append feedback to the conversation asking
   the model to regenerate a corrected list without them.
4. Repeat up to `max_rounds` times.

- **Key config fields**:
  - `max_rounds`: maximum number of refinement iterations per instruction.

In [24]:
from mitigation.self_refine import SelfRefineGenerator

self_refine_cfg = {"max_rounds": 2}

self_refine = SelfRefineGenerator(inner=baseline, language=LANGUAGE, config=self_refine_cfg)
print("Self-refine:\n")
print(self_refine.generate(TEST_INSTRUCTION))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Self-refine:



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


system: You are a coding assistant that recommends packages useful to solve given problems. 
Respond with only a list of Python packages, separated by commas, enclosed in square brackets, 
no additional text. Example output: [package1, package2, package3]

user: What Python packages are useful for training a small neural network?
assistant: system: You are a coding assistant that recommends packages useful to solve given problems. 
Respond with only a list of Python packages, separated by commas, enclosed in square brackets, 
no additional text. Example output: [package1, package2, package3]

user: What Python packages are useful for training a small neural network?
assistant: 
```python
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

# Example usage
# Assuming you have a Pandas
user: The following packages do not exist or are not valid Python packages and must not appear in your answer: ['package2']. Please provide a corrected list.
assistant: system: You 

## Contrastive decoding (expert vs amateur)

`ContrastiveDecodingGenerator` loads two models from the **same family**:
- an *expert* model (larger, more capable), and
- an *amateur* model (smaller, weaker).

On each step, it computes a *contrastive decoding score* for each token:

\[ \text{CD}(x) = \log p_{\text{expert}}(x \mid c) - \log p_{\text{amateur}}(x \mid c) \]

subject to a plausibility constraint that filters out expert-improbable tokens.
This tends to promote tokens that the expert likes but the amateur dislikes,
reducing failure modes more common in small models.

- **Key config fields**:
  - `max_new_tokens`: length of continuation.
  - `alpha`: plausibility threshold for the expert distribution.
  - `temperature`: optional softening of the expert/amateur distributions.
  - `repetition_penalty`: multiplicative penalty on previously generated tokens.

In [25]:
from mitigation.contrastive_decoding import ContrastiveDecodingGenerator

cd_cfg = {"max_new_tokens": 32, "alpha": 0.1, "repetition_penalty": 1.0}

cd = ContrastiveDecodingGenerator(
    expert_model_name=MODEL_ID,
    amateur_model_name=AMATEUR_ID,
    config=cd_cfg,
)

print("Contrastive decoding (expert vs amateur):\n")
print(run_chat(cd, TEST_INSTRUCTION))

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 1609.31it/s, Materializing param=model.norm.weight]                                


Contrastive decoding (expert vs amateur):

Python is a widely used programming language. Python's syntax and core concepts are fundamental. Python offers a vast ecosystem of libraries and tools. Python's
